# Partially Linear Models with `doubleml`

**Goal:** Estimate a causal effect using `DoubleMLPLR`, compare nuisance model choices, and plot confidence intervals.

**Time:** 15 minutes

You will run the full `doubleml` pipeline on a carbon price policy scenario,
compare Lasso / RF / GBM nuisance models, and interpret the output.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate: Carbon Price on Power Generation Mix

Treatment: carbon price policy change. Outcome: coal share of generation.
50 controls with nonlinear confounding.

In [ ]:
n = 2000
p = 50
true_theta = -0.5

X = np.random.randn(n, p)
col_names = [f'X{i}' for i in range(p)]

D = 0.5 * np.sin(X[:, 0]) + 0.3 * X[:, 1]**2 + 0.2 * X[:, 3] + np.random.randn(n) * 0.5
Y = (true_theta * D + 0.8 * np.exp(0.2 * X[:, 0])
     + 0.4 * X[:, 2] + 0.3 * X[:, 4] * X[:, 5] + np.random.randn(n) * 0.3)

df = pd.DataFrame(X, columns=col_names)
df['coal_share'] = Y
df['carbon_price_change'] = D

dml_data = DoubleMLData(df, y_col='coal_share',
                        d_cols='carbon_price_change', x_cols=col_names)
print(dml_data)
print(f'\nTrue effect: {true_theta}')

## Fit PLR with Gradient Boosting

Use gradient boosting for both nuisance functions. This is the recommended default.

In [ ]:
ml_l = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                  learning_rate=0.1, random_state=42)
ml_m = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                  learning_rate=0.1, random_state=42)

dml_plr = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m,
                       n_folds=5, score='partialling out')
dml_plr.fit()

print(dml_plr.summary)
print(f'\nTrue effect: {true_theta:.2f}')
print(f'Estimate:    {dml_plr.coef[0]:.4f}')
print(f'SE:          {dml_plr.se[0]:.4f}')
print(f'95% CI:      [{dml_plr.confint().iloc[0, 0]:.4f}, {dml_plr.confint().iloc[0, 1]:.4f}]')

## Compare Nuisance Model Choices

Fit PLR with three different ML models and compare treatment effect estimates.

In [ ]:
model_configs = [
    ('Lasso', LassoCV(cv=5, random_state=42), LassoCV(cv=5, random_state=42)),
    ('Random Forest',
     RandomForestRegressor(200, max_depth=10, random_state=42),
     RandomForestRegressor(200, max_depth=10, random_state=42)),
    ('Gradient Boosting',
     GradientBoostingRegressor(200, max_depth=5, learning_rate=0.1, random_state=42),
     GradientBoostingRegressor(200, max_depth=5, learning_rate=0.1, random_state=42)),
]

results = {}
for name, ml_l, ml_m in model_configs:
    dml = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
    dml.fit()
    ci = dml.confint()
    results[name] = {
        'coef': dml.coef[0], 'se': dml.se[0],
        'ci_low': ci.iloc[0, 0], 'ci_high': ci.iloc[0, 1]
    }
    print(f'{name:<20} theta={dml.coef[0]:+.4f}  SE={dml.se[0]:.4f}  '
          f'CI=[{ci.iloc[0,0]:+.4f}, {ci.iloc[0,1]:+.4f}]')

print(f'\nTrue effect: {true_theta}')

## Visualise: Forest Plot of Estimates

A forest plot shows point estimates with confidence intervals for each nuisance model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

names = list(results.keys())
coefs = [results[n]['coef'] for n in names]
ci_lows = [results[n]['ci_low'] for n in names]
ci_highs = [results[n]['ci_high'] for n in names]

y_pos = range(len(names))
xerr_low = [c - l for c, l in zip(coefs, ci_lows)]
xerr_high = [h - c for c, h in zip(coefs, ci_highs)]

ax.errorbar(coefs, y_pos, xerr=[xerr_low, xerr_high],
            fmt='o', capsize=6, markersize=10, linewidth=2, color='steelblue')
ax.axvline(x=true_theta, color='red', linestyle='--', linewidth=2,
           label=f'True = {true_theta}')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(names, fontsize=12)
ax.set_xlabel('Treatment Effect', fontsize=12)
ax.set_title('PLR Estimates by Nuisance Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('All models agree on the sign and magnitude -- result is robust.')

## Summary

**What you learned:**
1. `DoubleMLData` organises data into the Y, D, X structure
2. `DoubleMLPLR` handles cross-fitting, orthogonal scores, and inference automatically
3. Gradient boosting and random forests are good default nuisance models
4. If multiple nuisance models agree, the result is robust

**What is next:**
- **Module 06:** Interactive regression models for binary treatments (ATTE)
- **Module 07:** Instrumental variables with DML (PLIV)

**Key takeaway:** `doubleml` makes DML accessible. Focus on data quality and nuisance model selection.

In [ ]:
learning_objectives(["`DoubleMLData` organises data into the Y, D, X structure", "`DoubleMLPLR` handles cross-fitting, orthogonal scores, and inference automatically", "Gradient boosting and random forests are good default nuisance models", "If multiple nuisance models agree, the result is robust"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])